# Demo — EPINUC PTM colocalization on **TIF** runs (EpiVision)

The same analysis as the ND2 demo, over the flat TIF collections the EpiVision instrument
writes. `epinuc_tiff_loader.py` adapts them to the pipeline; nothing in the analysis changes.

One flowcell lane `chN` is one EPINUC channel: a **nucleosome template** (green, imaged at
cycle 1 only and reused as a static reference) plus a **red/blue antibody pair** imaged over 5
cycles. Positions → fields of view, cycles → time points, so the pipeline's cumulative
*new-only* counting runs across the antibody cycles.

Requirements: `epinuc_colocalization.py` and `epinuc_tiff_loader.py` next to this notebook, and
the images reachable (by default the `/Volumes/scBC` share). For the ND2 path see
`demo_epinuc_colocalization.ipynb`.

In [ ]:
import os

import epinuc_colocalization as ep
import epinuc_tiff_loader as tl

IMAGES_ROOT = "/Volumes/scBC/EpiVision/Images"   # TODO: folder holding the NUC### run folders
RUN         = "NUC388"                            # TODO: one run folder
OUTPUT_DIR  = "./demo_output"                     # per-lane CSVs land in <OUTPUT_DIR>/<run>/<lane>/
CONFIG      = "epinuc_config.json"                # tuned in the GUI: streamlit run epinuc_tiff_gui.py

# Order matters: apply_config restores every tuned knob (spot SNR, radii, ARTIFACT_*,
# TIF_CHANNEL_MAP), and configure_pipeline then pins the TIF-specific structural setup on top.
if os.path.isfile(CONFIG):
    ep.apply_config(ep.load_config(CONFIG))

# Pass artifact_masking through explicitly: configure_pipeline defaults it to True, which would
# otherwise silently override whatever the config just set. (epinuc_tiff_cli.py does the same.)
tl.configure_pipeline(artifact_masking=ep.ARTIFACT_MASKING)

print("spot SNR (k):    ", ep.SPOT_DETECTION_SNR)
print("artifact masking:", ep.ARTIFACT_MASKING)
print("channel map:     ", tl.current_tif_channel_map())

## 1. Index a run — lanes, cycles, lasers

`index_run` parses every TIF filename into one row
(`ch1_pos051_img0_Z1_R_cycle001_C001_000150_000.tif` → lane `ch1`, position 51, base `Z1`,
laser `R`, cycle 1) and stamps each file with its pipeline **role** from the current channel map.

In [ ]:
idx = tl.index_run(os.path.join(IMAGES_ROOT, RUN))

lanes = tl.lanes_in_run(idx)
LANE  = lanes[0]                                  # TODO: pick a lane

print(f"{len(idx)} files | lanes: {lanes}")
print(f"lasers present: {tl.lasers_in_run(idx)}")
print(f"{LANE}: cycles {tl.lane_cycles(idx, LANE)}, {len(tl.lane_positions(idx, LANE))} FOVs")
idx.head()

## 2. Channel → role

**The "channel" here is the laser letter in the filename** (`G` / `R` / `B`) — *not* the `chN`
prefix, which is the flowcell lane. Which laser carries which role is a property of how the run
was acquired, so it is selectable (the TIF analogue of `CHANNEL_MAP` for ND2s).

The default is `{"nucleosome": "G", "R_PTM": "R", "B_PTM": "B"}`. `check_tif_channel_map` shows
whether each role resolves against *this* run — `present=False` is the usual cause of an
all-blank channel downstream.

In [ ]:
tl.check_tif_channel_map(idx, tl.current_tif_channel_map())

In [ ]:
# If a run swapped the red/blue antibodies onto the other lasers, re-map them. Each role needs
# its own laser and 'nucleosome' must be assigned, or this raises rather than silently analysing
# one image under two names. Equivalent to the GUI's "Channel -> role" pickers, or the CLI's
# --channel-map nucleosome=G,R_PTM=B,B_PTM=R
#
# tl.configure_pipeline(tif_channel_map={"nucleosome": "G", "R_PTM": "B", "B_PTM": "R"})
# idx = tl.assign_roles(idx)      # re-role the index in place; no need to re-scan the folder

## 3. Quick run — one lane, two fields of view

`scenes=[0, 1]` keeps this fast. Each FOV's whole 5-cycle series stays inside one worker, so
the cumulative new-only counting is unaffected by the parallelism.

In [ ]:
summary = tl.run_channels_parallel(
    {RUN: idx},
    [(RUN, LANE)],           # (run, lane) jobs
    scenes=[0, 1],           # TODO: None for every FOV
    output_root=OUTPUT_DIR,
    n_jobs=ep.N_JOBS,        # -2 = all but one core; 1 = serial
)
summary

## 4. Full run & the "Results Format" table

- **Every run × every lane:** build the job list from the whole `IMAGES_ROOT` and pass
  `scenes=None`. Lanes are processed one at a time (bounding memory) while each lane's FOVs fan
  out over the cores.
- **Headless:** `python epinuc_tiff_cli.py --config epinuc_config.json --output per_run_output`
  does exactly this, plus the run-sheet annotation. Use `--list` to see each run's lasers, and
  `--channel-map ROLE=LASER,...` to override the laser assignment.
- **Tuning:** `streamlit run epinuc_tiff_gui.py` — assign the channels, tune the spot/bead/artifact
  knobs against live overlays, then press 💾 Save to write `epinuc_config.json`.

In [ ]:
results = tl.results_format_table(summary)
results

In [ ]:
# Full run over every run x lane, all FOVs (uncomment):
#
# run_index = {r: tl.index_run(os.path.join(IMAGES_ROOT, r))
#              for r in sorted(os.listdir(IMAGES_ROOT))
#              if os.path.isdir(os.path.join(IMAGES_ROOT, r))}
# jobs = [(r, lane) for r, ix in run_index.items() for lane in tl.lanes_in_run(ix)]
# all_summaries = tl.run_channels_parallel(run_index, jobs, scenes=None, output_root=OUTPUT_DIR)
# tl.results_format_table(all_summaries)